In [2]:
def generate_with_retry(topic, context, required_count, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        questions = generate_questions_for_topic(topic, context, required_count)
        
        # ADD THESE LOGS — find where it's actually failing
        print(f"[DEBUG] Topic: {topic}")
        print(f"[DEBUG] Attempt: {attempt}")
        print(f"[DEBUG] Context length: {len(context)} chars")
        print(f"[DEBUG] Required: {required_count}, Got: {len(questions)}")
        print(f"[DEBUG] Raw LLM output sample: {questions[:1]}")
        
        if len(questions) >= required_count:
            return questions[:required_count]
    
    # If you reach here — primary generation is broken, not slow
    print(f"[CRITICAL] Fallback triggered for {topic}. All {max_attempts} attempts failed.")
    return generate_fallback_questions(topic, required_count)

In [7]:
from neomodel import db

results, _ = db.cypher_query("""
    MATCH (t:Topic) 
    RETURN t.name AS name, labels(t) AS labels
    LIMIT 50
""")

print("=== ALL TOPIC NODES IN NEO4J ===")
for row in results:
    print(f"Name: {row[0]}, Labels: {row[1]}")

AuthError: {code: Neo.ClientError.Security.Unauthorized} {message: The client is unauthorized due to authentication failure.}

In [4]:
# CELL 2 — Cross-check your selected topics against what's in Neo4j

selected_topics = [
    "Algorithms", "Compiler Design", "Computer Networks",
    "Computer Organization and Architecture", "Databases",
    "Digital Logic", "Engineering Mathematics", "Operating System",
    "Programming and Data Structures", "Theory of Computation"
]

with driver.session() as session:
    print("=== TOPIC MATCH REPORT ===")
    for topic in selected_topics:
        # Exact match
        exact = session.run("""
            MATCH (t:Topic {name: $name}) RETURN count(t) AS cnt
        """, name=topic).single()["cnt"]
        
        # Fuzzy match
        fuzzy = session.run("""
            MATCH (t:Topic) 
            WHERE toLower(t.name) CONTAINS toLower($name)
            RETURN t.name AS found
            LIMIT 3
        """, name=topic[:6]).data()
        
        status = "✓ FOUND" if exact > 0 else "✗ MISSING"
        fuzzy_names = [r["found"] for r in fuzzy]
        print(f"  {status}  '{topic}'")
        if exact == 0:
            print(f"           Possible matches: {fuzzy_names}")

NameError: name 'driver' is not defined